# Llama room-to-spare + no-computation controls (EXP28 recipe)

STANDALONE. Run cell 1, RESTART the kernel, then run from cell 2 to the end. One 48 GB card,
roughly 35–45 minutes including about 19 GB of model downloads.

**The question.** Objection: "computation didn't transfer because sending the answer already
satisfied training, so the map never needed it." To answer it, a run needs both

1. **room to spare**: the map does NOT fit its own training set, so better transfer (including
   computation the recipient could finish) would still have lowered the training loss; and
2. **evidence that, despite that, only the answer transferred.**

**Maps** (all Llama-3.1-8B L17 → Llama-3.2-1B L11, k = 1 last-position graft):

| name | recipe | where it comes from |
|---|---|---|
| `exp28_3k` (seeds 0, 1) | 3,000 problems, 2 epochs | EXP28 baseline, `Full token position sweep/Llama fix` |
| `exp28_18k` (seeds 0, 1) | 18,000 problems, 2 epochs | EXP28 EXP-D, the paper's data-scale appendix (0.586) |
| `paper_6ep` (seed 0) | 3,000 problems, 6 epochs | the paper's main Llama map (0.278), `Llama_eval_EXP13-19` A2 |
| `const_3k`, `const_18k` | bias only (W = 0), same data and epochs | no-information baseline: one vector for every problem |

**Parts**
- **S** sanity: bins, training loss per epoch, conferral, shuffle, low-rank and erasure values
  reproduce the GitHub runs (`exp28_allpos_llama (3).json`, `consolidated_results (10).json`).
- **A** room to spare: conferral AND cross-entropy of the frozen trained map on its own training
  problems, split by whether the donor gets them right, next to the same numbers on the eval bin.
- **B** shuffle: graft another problem's donor state; does the recipient emit that problem's answer?
- **C** SVD truncation of the trained map (the paper's EXP10 method), on the eval bin AND training
  problems: how many of the 2,048 directions carry conferral, and what the rest were used for.
- **D** answer erasure from the donor state before the frozen map: EXP28 EXP-C verbatim (leading
  digit, uncentered), a centered version, and a new all-digit version. Random-direction controls,
  per-digit scoring, held-out linear and MLP probes, and an MLP baseline on the recipient's own state.
- **E** prompt swap per digit (the §6.2 test): does each digit follow the donor's problem or the
  recipient's prompt, against the recipient's own baselines?

**Not included: the probe bottleneck.** On Llama it can't be read cleanly. At 3k the readers decode
the whole answer only 2–15% of the time, and at 18k gold-label MLP readers compute answers
themselves (they read 40% of answers from the recipient's own state). Parts D and E test the same
question without that problem.

In [1]:
# === CELL 1: install (run once, then RESTART THE KERNEL) — EXP28's pins ===
!pip install -q "transformers==4.46.3" accelerate datasets numpy huggingface_hub hf_transfer
!pip uninstall -y torchvision torchaudio
print("deps installed — RESTART THE KERNEL now, then continue from CELL 2")


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Found existing installation: torchvision 0.19.1+cu124
Uninstalling torchvision-0.19.1+cu124:
  Successfully uninstalled torchvision-0.19.1+cu124
Found existing installation: torchaudio 2.4.1+cu124
Uninstalling torchaudio-2.4.1+cu124:
  Successfully uninstalled torchaudio-2.4.1+cu124
deps installed — RESTART THE KERNEL now, then continue from CELL 2


In [1]:
# === CELL 2: imports + RUN CONFIG ===
import os
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
import json, math, random, time, gc, collections, re
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import is_torch_available
assert is_torch_available(), "transformers does not see torch: re-run cell 1 and RESTART THE KERNEL"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | transformers {transformers.__version__} | cuda {torch.cuda.is_available()}")

SMOKE_TEST = False             # True = plumbing check with tiny sizes; numbers meaningless

# --- EXP28 config, Llama (copied from Stitch_EXP28_allpositions (12).ipynb, cell 2) ---
MODEL_D, MODEL_R, L_R, L_D = "meta-llama/Llama-3.1-8B", "meta-llama/Llama-3.2-1B", 11, 17
ARITH = dict(a_hi=30, b_hi=30, c_hi=40, ans_min=100, ans_max=999)
N_TRAIN, N_EVAL, N_TRAIN_10X = 3000, 2000, 18000
BATCH, TASK_EPOCHS, TASK_SEEDS, MAX_NEW, RIDGE_LAMBDA = 16, 2, [0, 1], 6, 1e3
PAPER_EPOCHS = 6               # Llama_eval_EXP13-19 TASK_EPOCHS (the paper's 0.278 map)
INLP_RANKS = [9, 27, 54, 108, 216, 360, 540]          # = EXP28 EXP-C
DIGIT_INLP_RANKS = [30, 90, 180, 360, 540]            # all-digit erasure: 30 directions per round
SVD_RANKS = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]   # + full
TRAIN_SUBSAMPLE = 800          # training problems per subset for the SVD sweep (full sets elsewhere)
CONST_STEPS = 2250             # Adam steps for both bias-only constants (= 2 epochs of 18k), so they differ only in data
RUN_18K = True
P_SHUFFLE_SEED, CHANCE_DRAWS = 1234, 20               # = Part D of Stitch_Llama_AnswerBottleneck
if SMOKE_TEST:
    N_TRAIN, N_EVAL, N_TRAIN_10X = 200, 150, 600
    INLP_RANKS, DIGIT_INLP_RANKS, SVD_RANKS = [9, 18], [30, 60], [1, 4, 16]
    TRAIN_SUBSAMPLE, CONST_STEPS = 100, 80

_space = sum(1 for a in range(2, ARITH["a_hi"] + 1) for b in range(2, ARITH["b_hi"] + 1)
             for c in range(1, ARITH["c_hi"] + 1) if ARITH["ans_min"] <= a * b + c <= ARITH["ans_max"])
assert N_TRAIN + N_EVAL + (N_TRAIN_10X if RUN_18K else 0) <= _space

OUT_JSON = "exp28_room_and_controls_llama" + ("_SMOKE" if SMOKE_TEST else "") + ".json"
RESULTS = {"_config": dict(donor=MODEL_D, recipient=MODEL_R, L_R=L_R, L_D=L_D, n_train=N_TRAIN,
                           n_train_10x=N_TRAIN_10X, epochs=TASK_EPOCHS, paper_epochs=PAPER_EPOCHS,
                           seeds=TASK_SEEDS, arith=ARITH, smoke=SMOKE_TEST)}
PER_ITEM = {}
def save(tag=""):
    with open(OUT_JSON, "w") as f:
        json.dump({**RESULTS, "per_item": PER_ITEM, "_last": tag}, f, indent=1, default=str)
    print(f"  [saved {OUT_JSON}{' after ' + tag if tag else ''}]")

# --- what the GitHub runs got (sanity targets) ---
REF = {
    "bins": dict(n_donor_solved=642, n_unsolvable=626, n_solvable=16),
    "exp28_3k_ce": {"seed0": [4.6765, 2.4077], "seed1": [4.7584, 2.5544]},
    "exp28_3k_first_seed0": 0.217, "exp28_3k_shuffle_seed0": 0.010,
    "exp28_3k_erasure_seed0": {9: 0.091, 27: 0.056, 54: 0.034, 108: 0.030, 216: 0.010, 360: 0.003, 540: 0.002},
    "exp28_3k_erasure_random_seed0": {9: 0.217, 27: 0.214, 54: 0.217, 108: 0.212, 216: 0.228, 360: 0.230, 540: 0.241},
    "exp28_18k_ce": {"seed0": [2.9734, 1.5806], "seed1": [2.9199, 1.5713]},
    "exp28_18k_first_seed0": 0.586,
    "paper_6ep_first": 0.278,
    "paper_6ep_lowrank": {1: 0.000, 2: 0.006, 4: 0.016, 8: 0.050, 16: 0.139, 32: 0.257},
    "swap_counts": {"far": (609, 598), "leading": (179, 159)},
    # the paper map (= task_3k_seed0) under the prompt swap, from llama_followup_results (F5)
    "paper_6ep_swap": {"far": {"ORIGINAL_y_whole": 0.266, "MISMATCH_y_whole": 0.201, "MISMATCH_y2_ones": 0.092},
                       "leading": {"ORIGINAL_y_whole": 0.233, "MISMATCH_y_whole": 0.182}},
}
SANITY = {}
def check(name, got, want, tol, warn=None):
    """PASS within tol; WARN within warn (a bf16 argmax tie or a different GPU can move a few items);
    CHECK beyond that."""
    d = abs(got - want)
    st = "PASS" if d <= tol else ("WARN" if warn is not None and d <= warn else "CHECK")
    SANITY[name] = {"github": want, "this_run": got, "status": st}
    print(f"  {st:5s} {name:44s} github {want}   this run {got}")
print("SMOKE_TEST =", SMOKE_TEST, "| device", DEVICE)

torch 2.4.1+cu124 | transformers 4.46.3 | cuda True
SMOKE_TEST = False | device cuda


In [ ]:
# === CELL 3: Hugging Face login (Llama is gated) ===
from huggingface_hub import login
login(os.environ.get("HF_TOKEN") or "")     # paste, run, then clear before sharing
print("logged in")

logged in


In [3]:
# === CELL 4: statistics helpers (= EXP28 cell 4) ===
BOOT_B = 2000
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),) * 3
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = (z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / d
    return p, max(0.0, c - h), min(1.0, c + h)
def wilson_bools(b, z=1.96):
    b = np.asarray(b, bool); return wilson(int(b.sum()), int(b.size), z)
_TC = {2: 12.706, 3: 4.303, 4: 3.182, 5: 2.776, 6: 2.571, 7: 2.447, 8: 2.365, 9: 2.306, 10: 2.262}
def across_seed_ci(v, alpha=0.05):
    v = np.asarray(v, float); k = len(v); m = float(v.mean())
    if k < 2: return (m, float("nan"), float("nan"))
    se = v.std(ddof=1) / math.sqrt(k); t = _TC.get(k, 1.96)
    return (m, m - t * se, m + t * se)
def fmt(tr): p, lo, hi = tr; return f"{p:.3f} [{lo:.3f}, {hi:.3f}]"
def rate(v): return round(float(np.mean(v)), 3) if len(v) else None
def mcnemar_exact(a, b):
    from fractions import Fraction
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    n01 = int((~a & b).sum()); n10 = int((a & ~b).sum()); n = n01 + n10
    if n == 0: return dict(n01=0, n10=0, p=1.0)
    k = min(n01, n10); tot = Fraction(0)
    for i in range(0, k + 1): tot += Fraction(math.comb(n, i))
    return dict(n01=n01, n10=n10, p=min(1.0, float(2 * tot / Fraction(2) ** n)))
print("stats ready")

stats ready


In [4]:
# === CELL 5: shared helpers (= EXP28 cell 5, last-position graft only) ===
PATCH_POS = -1
def _hid(o):  return o[0] if isinstance(o, tuple) else o
def _pack(o, h): return (h,) + tuple(o[1:]) if isinstance(o, tuple) else h
def left_pad(id_list, pad_id):
    L = max(t.numel() for t in id_list)
    ids = torch.full((len(id_list), L), pad_id, dtype=torch.long)
    m = torch.zeros((len(id_list), L), dtype=torch.long)
    for i, t in enumerate(id_list):
        t = t.flatten(); ids[i, L - t.numel():] = t; m[i, L - t.numel():] = 1
    return ids, m
def fit_ridge(X9, X2, lam=RIDGE_LAMBDA):
    mu9, mu2 = X9.mean(0), X2.mean(0)
    A, B = X9 - mu9, X2 - mu2
    W = torch.linalg.solve(A.T @ A + lam * torch.eye(A.shape[1]), A.T @ B)
    return mu9, mu2, W
FEWSHOT = ("2 + 5 = 7\n6 * 3 = 18\n4 * 7 + 2 = 30\n9 * 8 = 72\n"
           "3 * 4 + 5 = 17\n40 * 20 = 800\n")
def _aprompt(expr): return f"{FEWSHOT}{expr} ="
def _aencode(tok, expr, ans):
    p = tok(_aprompt(expr)).input_ids
    f = tok(_aprompt(expr) + " " + str(ans)).input_ids
    if f[:len(p)] != p or len(f) <= len(p): return None, None
    j = len(p)
    while j < len(f) and not any(ch.isdigit() for ch in tok.decode([f[j]])): j += 1
    if j >= len(f): return None, None
    return torch.tensor(f[:j]), f[j]
def gen_arith(tok, n, rng, exclude=None):
    exclude = exclude or set(); out, seen = [], set(); tries = 0
    while len(out) < n and tries < n * 300:
        tries += 1
        a = rng.randint(2, ARITH["a_hi"]); b = rng.randint(2, ARITH["b_hi"])
        c = rng.randint(1, ARITH["c_hi"])
        expr, ans = f"{a} * {b} + {c}", a * b + c
        if ARITH["ans_max"] is not None and ans > ARITH["ans_max"]: continue
        if ARITH["ans_min"] is not None and ans < ARITH["ans_min"]: continue
        if expr in seen or expr in exclude: continue
        seen.add(expr)
        ids, tok_id = _aencode(tok, expr, ans)
        if tok_id is None: continue
        out.append(dict(expr=expr, ans=ans, ids=ids, tok=tok_id))
    return out
_graft = {"vec": None}
def patch_vec_batch(_m, _i, o):
    h = _hid(o); vec = _graft["vec"]
    if vec is None or h.shape[1] <= 1 or h.shape[0] != vec.shape[0]: return o
    h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
    return _pack(o, h2)
def _digits(ans):
    s = str(int(ans)); assert len(s) == 3, ans
    return [int(ch) for ch in s]
print("helpers ready")

helpers ready


In [5]:
# === CELL 6: load both models; assert a shared tokenizer (= EXP28 cell 6) ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_R)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model_r = AutoModelForCausalLM.from_pretrained(
    MODEL_R, torch_dtype=torch.bfloat16, attn_implementation="eager",
    low_cpu_mem_usage=True).to(DEVICE).eval()
model_d = AutoModelForCausalLM.from_pretrained(
    MODEL_D, torch_dtype=torch.bfloat16, attn_implementation="eager",
    low_cpu_mem_usage=True).to(DEVICE).eval()
_t1 = AutoTokenizer.from_pretrained(MODEL_R); _t2 = AutoTokenizer.from_pretrained(MODEL_D)
_probe = "3 * 12 + 7 = 43\nThe answer is 256."
assert _t1(_probe).input_ids == _t2(_probe).input_ids, "tokenizer mismatch"
del _t1, _t2
print("loaded:", model_r.config.num_hidden_layers, "recipient layers,", model_d.config.num_hidden_layers, "donor layers")

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

loaded: 16 recipient layers, 32 donor layers


In [6]:
# === CELL 7: problems, states, bins (= EXP28 cell 7 without the all-position states) ===
D_D, D_R = model_d.config.hidden_size, model_r.config.hidden_size
@torch.inference_mode()
def states_last_and_top(model, layer, probs, batch=BATCH, gold_ce=None):
    """= EXP28. gold_ce=list collects the model's own CE on each problem's gold first token (same pass)."""
    acc, top = [], []
    for i in range(0, len(probs), batch):
        chunk = probs[i:i + batch]
        ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
        out = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        acc.append(out.hidden_states[layer + 1][:, -1, :].float().cpu())
        lg = out.logits[:, -1, :]
        top += lg.argmax(-1).cpu().tolist()
        if gold_ce is not None:
            gold_ce += F.cross_entropy(lg.float(), torch.tensor([p["tok"] for p in chunk], device=lg.device),
                                       reduction="none").cpu().tolist()
    return torch.cat(acc), top

t0 = time.time()
train = gen_arith(tokenizer, N_TRAIN, random.Random(0))
evalp = gen_arith(tokenizer, N_EVAL, random.Random(1), {p["expr"] for p in train})
print(f"arith: {len(train)} train, {len(evalp)} eval")
DONOR_CE_TR = []                                                   # donor's own CE on the gold token
X9t, d_top_tr = states_last_and_top(model_d, L_D, train, gold_ce=DONOR_CE_TR)   # EXP28 discarded d_top_tr
X9e, d_top = states_last_and_top(model_d, L_D, evalp)
keep = [i for i in range(len(evalp)) if d_top[i] == evalp[i]["tok"]]
evalp = [evalp[i] for i in keep]; X9e = X9e[keep]
X2t, r_top_tr = states_last_and_top(model_r, L_R, train)
X2e, r_top = states_last_and_top(model_r, L_R, evalp)
solvable = [r_top[i] == evalp[i]["tok"] for i in range(len(evalp))]
unsolv = [i for i in range(len(evalp)) if not solvable[i]]
solv = [i for i in range(len(evalp)) if solvable[i]]
RECON3 = fit_ridge(X9t, X2t)
print(f"  donor solves {len(evalp)} | recipient natively solves {len(solv)} | unsolvable bin n={len(unsolv)}  [{time.time()-t0:.0f}s]")
RESULTS["bins"] = dict(n_train=len(train), n_donor_solved=len(evalp), n_unsolvable=len(unsolv), n_solvable=len(solv))
print("SANITY (bins vs GitHub):")
for k, v in REF["bins"].items(): check(f"bins/{k}", RESULTS["bins"][k], v, 0 if not SMOKE_TEST else 10**9, warn=2)

if RUN_18K:   # = EXP28 EXP-D: Random(7), excluding the donor-solved eval expressions
    t0 = time.time()
    train10 = gen_arith(tokenizer, N_TRAIN_10X, random.Random(7), {p["expr"] for p in evalp})
    DONOR_CE_10 = []
    X9t10, d_top10 = states_last_and_top(model_d, L_D, train10, gold_ce=DONOR_CE_10)
    X2t10, r_top10 = states_last_and_top(model_r, L_R, train10)
    RECON18 = fit_ridge(X9t10, X2t10)
    print(f"  18k set: {len(train10)} problems  [{time.time()-t0:.0f}s]")
save("states")

arith: 3000 train, 2000 eval
  donor solves 642 | recipient natively solves 16 | unsolvable bin n=626  [36s]
SANITY (bins vs GitHub):
  PASS  bins/n_donor_solved                          github 642   this run 642
  PASS  bins/n_unsolvable                            github 626   this run 626
  PASS  bins/n_solvable                              github 16   this run 16
  18k set: 18000 problems  [136s]
  [saved exp28_room_and_controls_llama.json after states]


In [7]:
# === CELL 8: train the maps (= EXP28 train_map at k=1, generalised to any training set) ===
def train_map(items, X9tr, ridge, seeds, epochs, tag, bias_only=False, max_steps=None):
    """EXP28 recipe: W starts at the ridge map, b at the recipient mean, Adam lr 1e-3, CE on the
    first answer token, order random.Random(seed*100 + ep). bias_only=True holds W at 0 and trains b
    alone: one vector for every problem (the no-information baseline). max_steps stops early."""
    mu9_, mu2_, Wr_ = ridge
    mu9d_ = mu9_.to(DEVICE)
    maps, ce_log = [], {}
    model_r.requires_grad_(False)
    for seed in seeds:
        torch.manual_seed(seed); random.seed(seed)
        if bias_only:
            W = torch.zeros_like(Wr_).to(DEVICE)
        else:
            W = Wr_.clone().to(DEVICE).requires_grad_(True)
        b = mu2_.clone().to(DEVICE).requires_grad_(True)
        opt = torch.optim.Adam([b] if bias_only else [W, b], lr=1e-3)
        hook = model_r.model.layers[L_R].register_forward_hook(patch_vec_batch)
        idx = list(range(len(items))); ep_ce = []; steps = 0
        try:
            for ep in range(epochs):
                if max_steps is not None and steps >= max_steps: break
                random.Random(seed * 100 + ep).shuffle(idx); _run, _nb = 0.0, 0
                for s in range(0, len(idx), BATCH):
                    if max_steps is not None and steps >= max_steps: break
                    steps += 1
                    sub = idx[s:s + BATCH]
                    ids, m = left_pad([items[j]["ids"] for j in sub], tokenizer.pad_token_id)
                    _graft["vec"] = (X9tr[sub].to(DEVICE) - mu9d_) @ W + b
                    lg = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                    tgt = torch.tensor([items[j]["tok"] for j in sub], device=DEVICE)
                    loss = F.cross_entropy(lg, tgt)
                    opt.zero_grad(); loss.backward(); opt.step()
                    _run += float(loss.detach()); _nb += 1
                ep_ce.append(round(_run / max(1, _nb), 4))
                print(f"    {tag} seed{seed} ep{ep}: mean CE {ep_ce[-1]:.4f}")
        finally:
            hook.remove(); _graft["vec"] = None
        maps.append(dict(W=W.detach(), b=b.detach(), mu=mu9d_, W0=(None if bias_only else Wr_.to(DEVICE)),
                         steps=steps, train_set_size=len(items)))
        ce_log[f"seed{seed}"] = ep_ce
    model_r.requires_grad_(True)
    return maps, ce_log

def fn_of(M, W=None):
    """Map as a function of donor states. W overrides the weights (SVD truncation); bias kept."""
    W_ = M["W"] if W is None else W
    return lambda x9, W_=W_, b=M["b"], mu=M["mu"]: (x9.to(DEVICE) - mu) @ W_ + b

t0 = time.time()
MAPS, CE = {}, {}
_m, CE["exp28_3k"] = train_map(train, X9t, RECON3, TASK_SEEDS, TASK_EPOCHS, "exp28_3k")
MAPS["exp28_3k_s0"], MAPS["exp28_3k_s1"] = _m[0], _m[1]
_m, CE["paper_6ep"] = train_map(train, X9t, RECON3, [0], PAPER_EPOCHS, "paper_6ep"); MAPS["paper_6ep"] = _m[0]
_ce_ep = lambda n: math.ceil(CONST_STEPS / math.ceil(n / BATCH))      # epochs that cover CONST_STEPS steps
_m, CE["const_3k"] = train_map(train, X9t, RECON3, [0], _ce_ep(len(train)), "const_3k", bias_only=True,
                               max_steps=CONST_STEPS); MAPS["const_3k"] = _m[0]
if RUN_18K:
    _m, CE["exp28_18k"] = train_map(train10, X9t10, RECON18, TASK_SEEDS, TASK_EPOCHS, "exp28_18k")
    MAPS["exp28_18k_s0"], MAPS["exp28_18k_s1"] = _m[0], _m[1]
    _m, CE["const_18k"] = train_map(train10, X9t10, RECON18, [0], _ce_ep(len(train10)), "const_18k", bias_only=True,
                                    max_steps=CONST_STEPS)
    MAPS["const_18k"] = _m[0]
RESULTS["training_ce_running_mean_per_epoch"] = CE
RESULTS["adam_steps"] = {k: v["steps"] for k, v in MAPS.items()}
print(f"  trained {list(MAPS)} | Adam steps {RESULTS['adam_steps']}  [{time.time()-t0:.0f}s]")

print("SANITY (training loss vs GitHub EXP28):")
for sd in ("seed0", "seed1"):
    for ep in range(min(TASK_EPOCHS, len(REF["exp28_3k_ce"][sd]))):     # EXP28 logged 2 epochs
        check(f"exp28_3k CE {sd} ep{ep}", CE["exp28_3k"][sd][ep], REF["exp28_3k_ce"][sd][ep], 0.03, warn=0.10)
        if RUN_18K: check(f"exp28_18k CE {sd} ep{ep}", CE["exp28_18k"][sd][ep], REF["exp28_18k_ce"][sd][ep], 0.03, warn=0.10)
# the paper map's first two epochs are the same training run as exp28_3k seed 0 (same seed, order, init)
for ep in range(min(2, TASK_EPOCHS, PAPER_EPOCHS)):
    check(f"paper_6ep CE ep{ep} == exp28_3k seed0", CE["paper_6ep"]["seed0"][ep], CE["exp28_3k"]["seed0"][ep], 0.005, warn=0.03)
save("training")

    exp28_3k seed0 ep0: mean CE 4.6765
    exp28_3k seed0 ep1: mean CE 2.4077
    exp28_3k seed1 ep0: mean CE 4.7584
    exp28_3k seed1 ep1: mean CE 2.5544
    paper_6ep seed0 ep0: mean CE 4.6765
    paper_6ep seed0 ep1: mean CE 2.4077
    paper_6ep seed0 ep2: mean CE 1.6074
    paper_6ep seed0 ep3: mean CE 1.1533
    paper_6ep seed0 ep4: mean CE 0.7591
    paper_6ep seed0 ep5: mean CE 0.5805
    const_3k seed0 ep0: mean CE 5.6846
    const_3k seed0 ep1: mean CE 4.9334
    const_3k seed0 ep2: mean CE 4.4901
    const_3k seed0 ep3: mean CE 4.1850
    const_3k seed0 ep4: mean CE 3.9396
    const_3k seed0 ep5: mean CE 3.7380
    const_3k seed0 ep6: mean CE 3.5759
    const_3k seed0 ep7: mean CE 3.4484
    const_3k seed0 ep8: mean CE 3.3573
    const_3k seed0 ep9: mean CE 3.2975
    const_3k seed0 ep10: mean CE 3.2029
    const_3k seed0 ep11: mean CE 3.1433
    exp28_18k seed0 ep0: mean CE 2.9734
    exp28_18k seed0 ep1: mean CE 1.5806
    exp28_18k seed1 ep0: mean CE 2.9199
    exp28_18k 

In [8]:
# === CELL 9: scoring helpers ===
@torch.inference_mode()
def run_tops(fn, items, X9rows=None, want_logits_for=None):
    """First-token argmax on each item's prompt with fn(donor state) grafted (fn=None: no graft).
    want_logits_for: list of target tokens -> also returns per-item CE against them."""
    if fn is not None: assert X9rows.shape[0] == len(items)
    h = model_r.model.layers[L_R].register_forward_hook(patch_vec_batch) if fn is not None else None
    tops, ces = [], []
    try:
        for i in range(0, len(items), BATCH):
            chunk = items[i:i + BATCH]
            if fn is not None:
                v = fn(X9rows[i:i + len(chunk)]); assert v.shape[0] == len(chunk)
                _graft["vec"] = v
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            lg = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
            tops += lg.argmax(-1).cpu().tolist()
            if want_logits_for is not None:
                tgt = torch.tensor(want_logits_for[i:i + len(chunk)], device=DEVICE)
                ces += F.cross_entropy(lg, tgt, reduction="none").cpu().tolist()
    finally:
        if h is not None: h.remove()
        _graft["vec"] = None
    return (tops, ces) if want_logits_for is not None else tops

def confer_ce(fn, items, X9rows):
    """Conferral (top-1 == gold), mean CE of the gold token, and the top-1 tokens; frozen map."""
    tops, ces = run_tops(fn, items, X9rows, want_logits_for=[p["tok"] for p in items])
    ok = [t == p["tok"] for t, p in zip(tops, items)]
    return ok, (float(np.mean(ces)) if ces else float("nan")), tops

def _dig_of(t, n):
    """Digit n (0 hundreds, 1 tens, 2 ones) of a top-1 token if it is a 3-digit number, else None."""
    s = tokenizer.decode([t]).strip()
    return int(s[n]) if (len(s) == 3 and s.isdigit()) else None
def digit_rates(tops, items):
    """Per-digit agreement of each output with the item's own gold answer."""
    return {nm: rate([_dig_of(t, n) == _digits(p["ans"])[n] for t, p in zip(tops, items)])
            for n, nm in enumerate(("hundreds", "tens", "ones"))}
def _twin(expr):
    a, b, c = (int(x) for x in re.match(r"^(\d+) \* (\d+) \+ (\d+)$", expr).groups())
    return None if a == b else f"{b} * {a} + {c}"

BIN_ITEMS = [evalp[j] for j in unsolv]
X9BIN = X9e[unsolv]
def train_subsets(items, d_top_, r_top_):
    gold = [p["tok"] for p in items]
    return {"all": list(range(len(items))),
            "donor_right_recipient_wrong (bin-like)": [i for i in range(len(items)) if d_top_[i] == gold[i] and r_top_[i] != gold[i]],
            "donor_wrong_recipient_wrong": [i for i in range(len(items)) if d_top_[i] != gold[i] and r_top_[i] != gold[i]],
            "recipient_right": [i for i in range(len(items)) if r_top_[i] == gold[i]]}
SUB3 = train_subsets(train, d_top_tr, r_top_tr)
SUB18 = train_subsets(train10, d_top10, r_top10) if RUN_18K else None
TRAIN_OF = {"exp28_3k_s0": (train, X9t, SUB3), "exp28_3k_s1": (train, X9t, SUB3),
            "paper_6ep": (train, X9t, SUB3), "const_3k": (train, X9t, SUB3)}
DTOP_OF = {k: (d_top_tr, DONOR_CE_TR) for k in TRAIN_OF}
if RUN_18K:
    TRAIN_OF.update({"exp28_18k_s0": (train10, X9t10, SUB18), "exp28_18k_s1": (train10, X9t10, SUB18),
                     "const_18k": (train10, X9t10, SUB18)})
    DTOP_OF.update({k: (d_top10, DONOR_CE_10) for k in ("exp28_18k_s0", "exp28_18k_s1", "const_18k")})
print({k: len(v) for k, v in SUB3.items()}, "(3k training subsets)")
if RUN_18K: print({k: len(v) for k, v in SUB18.items()}, "(18k training subsets)")
# eval-bin problems whose operand-swap twin (b*a+c, same answer) is one of the map's training problems
def twin_split(train_items):
    ex = {p["expr"] for p in train_items}
    tw = [k for k, p in enumerate(BIN_ITEMS) if _twin(p["expr"]) in ex]
    return tw, [k for k in range(len(BIN_ITEMS)) if k not in set(tw)]
TWIN3 = twin_split(train)
TWIN18 = twin_split(train10) if RUN_18K else None
print(f"eval-bin problems with their twin in training: 3k {len(TWIN3[0])}/{len(BIN_ITEMS)}"
      + (f", 18k {len(TWIN18[0])}/{len(BIN_ITEMS)}" if RUN_18K else ""))

print("SANITY (eval-bin conferral vs GitHub):")
BIN_OK = {}
for name, M in MAPS.items():
    BIN_OK[name] = run_tops(fn_of(M), BIN_ITEMS, X9BIN)
    BIN_OK[name] = [t == p["tok"] for t, p in zip(BIN_OK[name], BIN_ITEMS)]
check("exp28_3k first seed0", rate(BIN_OK["exp28_3k_s0"]), REF["exp28_3k_first_seed0"], 0.010, warn=0.025)
check("paper_6ep first seed0", rate(BIN_OK["paper_6ep"]), REF["paper_6ep_first"], 0.015, warn=0.03)
if RUN_18K: check("exp28_18k first seed0", rate(BIN_OK["exp28_18k_s0"]), REF["exp28_18k_first_seed0"], 0.015, warn=0.03)
PER_ITEM["eval_bin_conferral"] = {k: [int(x) for x in v] for k, v in BIN_OK.items()}
save("scoring")

{'all': 3000, 'donor_right_recipient_wrong (bin-like)': 914, 'donor_wrong_recipient_wrong': 2035, 'recipient_right': 51} (3k training subsets)
{'all': 18000, 'donor_right_recipient_wrong (bin-like)': 5096, 'donor_wrong_recipient_wrong': 12632, 'recipient_right': 272} (18k training subsets)
eval-bin problems with their twin in training: 3k 81/626, 18k 400/626
SANITY (eval-bin conferral vs GitHub):
  PASS  exp28_3k first seed0                         github 0.217   this run 0.217
  PASS  paper_6ep first seed0                        github 0.278   this run 0.278
  PASS  exp28_18k first seed0                        github 0.586   this run 0.586
  [saved exp28_room_and_controls_llama.json after scoring]


In [15]:
# === CELL 10: PART A — room to spare: the frozen trained map on its OWN training problems ===
# Conferral and mean CE on training problems (split by whether the donor / recipient get them right)
# next to the eval bin. The bin is donor-right / recipient-wrong, so compare it with the bin-like
# training subset: same filter, the only difference is whether the map trained on them.
# On donor-wrong problems two extra numbers: how often the stitch emits the DONOR's own (wrong) top-1,
# and the donor's own CE on the gold token: the loss a map that passed on exactly the donor's output
# distribution would get. A map below that donor CE is fitting labels the donor's output doesn't have.
t0 = time.time()
A = {}
for name, M in MAPS.items():
    items, X9tr, sub = TRAIN_OF[name]; dtop, dce = DTOP_OF[name]
    row = {}
    for sname, ks in sub.items():
        if not ks: continue
        ok, ce, tops = confer_ce(fn_of(M), [items[i] for i in ks], X9tr[ks])
        r_ = {"n": len(ks), "conferral": fmt(wilson_bools(ok)), "mean_CE": round(ce, 4)}
        if sname == "donor_wrong_recipient_wrong":
            r_["emits_donors_own_wrong_answer"] = fmt(wilson_bools([t == dtop[i] for t, i in zip(tops, ks)]))
            r_["donor_own_CE_on_gold"] = round(float(np.mean([dce[i] for i in ks])), 4)
        if sname == "donor_right_recipient_wrong (bin-like)":
            r_["donor_own_CE_on_gold"] = round(float(np.mean([dce[i] for i in ks])), 4)
        row[f"train/{sname}"] = r_
    ok, ce, tops = confer_ce(fn_of(M), BIN_ITEMS, X9BIN)
    row["eval_bin"] = {"n": len(BIN_ITEMS), "conferral": fmt(wilson_bools(ok)), "mean_CE": round(ce, 4),
                       "per_digit": digit_rates(tops, BIN_ITEMS)}
    tw, nt = TWIN18 if items is not train else TWIN3
    row["eval_bin/twin_in_training"] = {"n": len(tw), "conferral": fmt(wilson_bools([ok[k] for k in tw])) if tw else None}
    row["eval_bin/no_twin_in_training"] = {"n": len(nt), "conferral": fmt(wilson_bools([ok[k] for k in nt])) if nt else None}
    A[name] = row
    print(f"\n  [{name}]  (Adam steps {M['steps']})")
    for k, v in row.items():
        extra = {kk: vv for kk, vv in v.items() if kk not in ("n", "conferral", "mean_CE")}
        print(f"     {k:44s} n={v['n']:6d}  conferral {v['conferral']}  mean CE {v.get('mean_CE', '-')}  {extra if extra else ''}")
RESULTS["A_room_to_spare"] = A
print(f"  [{time.time()-t0:.0f}s]"); save("A")


  [exp28_3k_s0]  (Adam steps 376)
     train/all                                    n=  3000  conferral 0.650 [0.633, 0.667]  mean CE 1.3779  
     train/donor_right_recipient_wrong (bin-like) n=   914  conferral 0.748 [0.719, 0.775]  mean CE 0.9756  {'donor_own_CE_on_gold': 0.879}
     train/donor_wrong_recipient_wrong            n=  2035  conferral 0.606 [0.584, 0.627]  mean CE 1.5662  {'emits_donors_own_wrong_answer': '0.027 [0.021, 0.035]', 'donor_own_CE_on_gold': 3.0755}
     train/recipient_right                        n=    51  conferral 0.667 [0.530, 0.780]  mean CE 1.0715  
     eval_bin                                     n=   626  conferral 0.217 [0.187, 0.251]  mean CE 2.8573  {'per_digit': {'hundreds': 0.919, 'tens': 0.313, 'ones': 0.588}}
     eval_bin/twin_in_training                    n=    81  conferral 0.432 [0.330, 0.541]  mean CE -  
     eval_bin/no_twin_in_training                 n=   545  conferral 0.185 [0.155, 0.220]  mean CE -  

  [exp28_3k_s1]  (Adam step

In [16]:
# === CELL 11: PART B — shuffle control (= EXP28's length-matched derangement, seed 0) ===
def length_matched_shuffle(idxs, seed=0):
    rng = random.Random(seed); by_len = {}
    for j in idxs: by_len.setdefault(evalp[j]["ids"].numel(), []).append(j)
    out = {}
    for Lk, group in by_len.items():
        if len(group) < 2: out[group[0]] = group[0]; continue
        perm = group[:]
        for _ in range(50):
            rng.shuffle(perm)
            if all(a != b for a, b in zip(group, perm)): break
        out.update(dict(zip(group, perm)))
    return [out[j] for j in idxs], sum(1 for j in idxs if out[j] == j)
SH, n_fixed = length_matched_shuffle(unsolv, seed=0)
B = {"n_fixed_points": n_fixed}
for name, M in MAPS.items():
    tops = run_tops(fn_of(M), BIN_ITEMS, X9e[SH])
    own = [t == p["tok"] for t, p in zip(tops, BIN_ITEMS)]
    partner = [t == evalp[s]["tok"] for t, s in zip(tops, SH)]
    B[name] = {"own_answer": fmt(wilson_bools(own)), "partner_answer": fmt(wilson_bools(partner)),
               "unshuffled": fmt(wilson_bools(BIN_OK[name]))}
    print(f"  {name:14s} unshuffled {B[name]['unshuffled']} | shuffled: own answer {B[name]['own_answer']}, "
          f"the OTHER problem's answer {B[name]['partner_answer']}")
RESULTS["B_shuffle"] = B
print("SANITY (shuffle vs GitHub):")
check("exp28_3k shuffle seed0", float(B["exp28_3k_s0"]["own_answer"].split()[0]), REF["exp28_3k_shuffle_seed0"], 0.008, warn=0.016)
save("B")

  exp28_3k_s0    unshuffled 0.217 [0.187, 0.251] | shuffled: own answer 0.010 [0.004, 0.021], the OTHER problem's answer 0.158 [0.132, 0.189]
  exp28_3k_s1    unshuffled 0.246 [0.214, 0.281] | shuffled: own answer 0.003 [0.001, 0.012], the OTHER problem's answer 0.158 [0.132, 0.189]
  paper_6ep      unshuffled 0.278 [0.244, 0.314] | shuffled: own answer 0.002 [0.000, 0.009], the OTHER problem's answer 0.214 [0.184, 0.248]
  const_3k       unshuffled 0.198 [0.169, 0.231] | shuffled: own answer 0.198 [0.169, 0.231], the OTHER problem's answer 0.005 [0.002, 0.014]
  exp28_18k_s0   unshuffled 0.586 [0.547, 0.624] | shuffled: own answer 0.003 [0.001, 0.012], the OTHER problem's answer 0.532 [0.493, 0.571]
  exp28_18k_s1   unshuffled 0.565 [0.526, 0.604] | shuffled: own answer 0.002 [0.000, 0.009], the OTHER problem's answer 0.510 [0.470, 0.549]
  const_18k      unshuffled 0.204 [0.175, 0.238] | shuffled: own answer 0.204 [0.175, 0.238], the OTHER problem's answer 0.003 [0.001, 0.012]
SANITY

In [17]:
# === CELL 12: PART C — SVD truncation of the trained map (= EXP10: truncate W, keep the bias) ===
# Two truncations, each on the eval bin AND on training problems (same sampled training problems for
# every map trained on the same set):
#   'W'  : truncate the whole trained W (= EXP10, the paper's table)
#   'dW' : keep the ridge starting map and truncate only the learned update dW = W - W_ridge. The tail
#          of W is mostly ridge (itself fit on training states), so dW is where "what training added" lives.
# Compare the SHARE of full-rank conferral each rank keeps on training vs eval: if the tail directions
# carried a general computation, cutting them would cost eval about as much as training; if they were
# memorising training problems, cutting them costs training far more.
t0 = time.time()
C = {}
_SAMPLES = {}
for name in [n for n in ("paper_6ep", "exp28_3k_s0", "exp28_18k_s0") if n in MAPS]:
    M = MAPS[name]; items, X9tr, sub = TRAIN_OF[name]
    if id(items) not in _SAMPLES:
        _rs = random.Random(0)
        _SAMPLES[id(items)] = {k: (_rs.sample(sub[k], min(TRAIN_SUBSAMPLE, len(sub[k]))) if sub[k] else [])
                               for k in ("donor_right_recipient_wrong (bin-like)", "donor_wrong_recipient_wrong")}
    subs = _SAMPLES[id(items)]
    C[name] = {"n_train_sampled": {k: len(v) for k, v in subs.items()}}
    for mode in ("W", "dW"):
        base = M["W"].float() if mode == "W" else (M["W"] - M["W0"]).float()
        U, S, Vh = torch.linalg.svd(base, full_matrices=False)
        energy = (S.pow(2).cumsum(0) / S.pow(2).sum()).tolist()
        rows = {}
        for r in [r for r in SVD_RANKS if r < S.numel()] + [S.numel()]:
            Wt = (U[:, :r] * S[:r]) @ Vh[:r]
            if mode == "dW": Wt = M["W0"].float() + Wt
            f = fn_of(M, Wt)
            tb = run_tops(f, BIN_ITEMS, X9BIN)
            row = {"energy": round(energy[r - 1], 3), "eval_bin": rate([t == p["tok"] for t, p in zip(tb, BIN_ITEMS)])}
            for k, ks in subs.items():
                if ks:
                    tt = run_tops(f, [items[i] for i in ks], X9tr[ks])
                    row[f"train/{k}"] = rate([t == items[i]["tok"] for t, i in zip(tt, ks)])
            rows["full" if r == S.numel() else r] = row
        full = rows["full"]
        for r, row in rows.items():      # share of full-rank conferral kept, per set
            row["share_of_full_kept"] = {k: (round(v / full[k], 3) if isinstance(v, float) and full.get(k) else None)
                                         for k, v in row.items() if k not in ("energy", "share_of_full_kept")}
        C[name][f"truncate_{mode}"] = rows
        for r, row in rows.items():
            print(f"  [{name} / truncate {mode}] rank {r!s:>5}: eval {row['eval_bin']} | train bin-like "
                  f"{row.get('train/donor_right_recipient_wrong (bin-like)')} | train donor-wrong "
                  f"{row.get('train/donor_wrong_recipient_wrong')} | kept {row['share_of_full_kept']}")
RESULTS["C_svd_truncation"] = C
if "paper_6ep" in C:
    print("SANITY (low-rank truncation of the paper map vs GitHub EXP10):")
    for r, v in REF["paper_6ep_lowrank"].items():
        if r in C["paper_6ep"]["truncate_W"]:
            check(f"paper_6ep rank {r}", C["paper_6ep"]["truncate_W"][r]["eval_bin"], v, 0.02, warn=0.04)
print(f"  [{time.time()-t0:.0f}s]"); save("C")

  [paper_6ep / truncate W] rank     1: eval 0.0 | train bin-like 0.001 | train donor-wrong 0.001 | kept {'eval_bin': 0.0, 'train/donor_right_recipient_wrong (bin-like)': 0.001, 'train/donor_wrong_recipient_wrong': 0.001}
  [paper_6ep / truncate W] rank     2: eval 0.006 | train bin-like 0.01 | train donor-wrong 0.001 | kept {'eval_bin': 0.022, 'train/donor_right_recipient_wrong (bin-like)': 0.011, 'train/donor_wrong_recipient_wrong': 0.001}
  [paper_6ep / truncate W] rank     4: eval 0.016 | train bin-like 0.033 | train donor-wrong 0.013 | kept {'eval_bin': 0.058, 'train/donor_right_recipient_wrong (bin-like)': 0.035, 'train/donor_wrong_recipient_wrong': 0.015}
  [paper_6ep / truncate W] rank     8: eval 0.05 | train bin-like 0.099 | train donor-wrong 0.052 | kept {'eval_bin': 0.18, 'train/donor_right_recipient_wrong (bin-like)': 0.105, 'train/donor_wrong_recipient_wrong': 0.059}
  [paper_6ep / truncate W] rank    16: eval 0.139 | train bin-like 0.253 | train donor-wrong 0.185 | kept {

In [18]:
# === CELL 13: PART D — answer erasure from the DONOR state before the frozen map ===
# Three variants, each with a random-direction control of the same rank:
#   D1  = EXP28 EXP-C verbatim: leading-digit probe directions projected out of the UNCENTERED donor
#         state (sanity target). Caveat: projecting uncentered states also shifts the state's mean along
#         the erased directions, so part of a collapse can be an off-distribution shift. Reported:
#         |mu (I - P)| for answer vs random erasure.
#   D1c = the same, CENTERED: probes fit on x - mu, erased state mu + (x - mu) P^T. No mean shift.
#   D2c = NEW, centered, all three digits: hundreds, tens and ones probes (30 directions per round).
# Scored on the eval bin: whole-answer conferral AND each output digit vs the gold digit (so we see
# whether tens and ones still get through once the answer directions are gone).
# Probe checks at the deepest rank, all HELD OUT (fit on training donor states, scored on the bin):
#   linear probe before/after; MLP probe (probe_mlp) before/after; and the same MLP on the RECIPIENT's
#   own L11 state, the "computes it from the operands" baseline. Leftover answer information is only
#   indicated if the MLP after erasure is well above BOTH the linear probe after and the recipient baseline.
t0 = time.time()
ERASE_MAPS = [n for n in ("exp28_3k_s0", "paper_6ep", "exp28_18k_s0") if n in MAPS]
def _fd(x): return int(str(abs(int(round(x))))[0])
ytr = (torch.tensor([_fd(p["ans"]) for p in train]) - 1).to(DEVICE)
ydig = torch.tensor([_digits(p["ans"]) for p in train], device=DEVICE)          # [n, 3]
ydig_bin = torch.tensor([_digits(p["ans"]) for p in BIN_ITEMS], device=DEVICE)
ytr_bin = (torch.tensor([_fd(p["ans"]) for p in BIN_ITEMS]) - 1).to(DEVICE)
MU3 = X9t.mean(0).to(DEVICE)

def fit_probe(X, y, steps=300, seed=0):                 # = EXP28
    torch.manual_seed(seed)
    lin = torch.nn.Linear(X.shape[1], 9).to(DEVICE)
    opt = torch.optim.Adam(lin.parameters(), lr=1e-3)
    Xd = X.to(DEVICE).clone()      # clone: states come from inference mode and must be autograd-safe
    for _ in range(steps):
        loss = F.cross_entropy(lin(Xd), y)
        opt.zero_grad(); loss.backward(); opt.step()
    return lin
def fit_digit_probe(X, Y, steps=300, seed=0):           # three 10-way heads, summed CE
    torch.manual_seed(seed)
    lin = torch.nn.Linear(X.shape[1], 30).to(DEVICE)
    opt = torch.optim.Adam(lin.parameters(), lr=1e-3)
    Xd = X.to(DEVICE).clone()      # clone: states come from inference mode and must be autograd-safe
    for _ in range(steps):
        z = lin(Xd); loss = sum(F.cross_entropy(z[:, 10*k:10*k+10], Y[:, k]) for k in range(3))
        opt.zero_grad(); loss.backward(); opt.step()
    return lin
def heldout_probe_acc(Xtr_, Xte_, kind):
    """Linear probe fit on training donor states, scored on the eval bin. kind 'lead' or 'digits'."""
    if kind == "lead":
        lin = fit_probe(Xtr_, ytr, seed=4242)
        with torch.no_grad(): return round(float((lin(Xte_.to(DEVICE)).argmax(1) == ytr_bin).float().mean()), 3)
    lin = fit_digit_probe(Xtr_, ydig, seed=4242)
    with torch.no_grad():
        z = lin(Xte_.to(DEVICE))
        return [round(float((z[:, 10*k:10*k+10].argmax(1) == ydig_bin[:, k]).float().mean()), 3) for k in range(3)]
def mlp_heldout_acc(Xtr_, Ytr_, Xte_, Yte_, n_cls):
    """probe_mlp (Followup3_MLPprobe_Llama): standardise, Linear(d,256)-GELU-Linear, AdamW 1e-3, wd 1e-3,
    800 steps; fit on training states, scored on eval-bin states."""
    Xtr_ = Xtr_.float().to(DEVICE); Xte_ = Xte_.float().to(DEVICE)
    mu, sd = Xtr_.mean(0), Xtr_.std(0).clamp_min(1e-6)
    torch.manual_seed(0)
    net = nn.Sequential(nn.Linear(Xtr_.shape[1], 256), nn.GELU(), nn.Linear(256, n_cls)).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-3)
    A_ = (Xtr_ - mu) / sd
    for _ in range(800 if not SMOKE_TEST else 50):
        opt.zero_grad(); F.cross_entropy(net(A_), Ytr_).backward(); opt.step()
    with torch.no_grad(): return round(float((net((Xte_ - mu) / sd).argmax(1) == Yte_).float().mean()), 3)
def mlp_suite(Xtr_, Xte_):
    out = {"leading_digit": mlp_heldout_acc(Xtr_, ytr, Xte_, ytr_bin, 9)}
    for k, nm in enumerate(("hundreds", "tens", "ones")):
        out[nm] = mlp_heldout_acc(Xtr_, ydig[:, k], Xte_, ydig_bin[:, k], 10)
    return out

def _erased(name, Xproj_bin):
    t = run_tops(fn_of(MAPS[name]), BIN_ITEMS, Xproj_bin.cpu())
    return [x == p["tok"] for x, p in zip(t, BIN_ITEMS)], digit_rates(t, BIN_ITEMS)

D = {"unerased": {}}
for name in ERASE_MAPS:
    ok, dr = _erased(name, X9BIN.to(DEVICE))
    D["unerased"][name] = {"whole": rate(ok), "per_digit": dr}
D["probes_before"] = {"donor_linear_lead": heldout_probe_acc(X9t, X9BIN, "lead"),
                      "donor_linear_digits": heldout_probe_acc(X9t, X9BIN, "digits"),
                      "donor_mlp": mlp_suite(X9t, X9BIN),
                      "recipient_own_L11_mlp (operands baseline)": mlp_suite(X2t, X2e[unsolv])}
print("  unerased:", D["unerased"]); print("  probes before erasure (held-out on bin):", D["probes_before"])

EYE = torch.eye(D_D, device=DEVICE)
for variant, ranks in (("D1_leading_digit_uncentered_EXP28", INLP_RANKS),
                       ("D1c_leading_digit_centered", INLP_RANKS),
                       ("D2c_all_digits_centered", DIGIT_INLP_RANKS)):
    centered = variant.startswith(("D1c", "D2c")); digits = variant.startswith("D2c")
    Xtr = (X9t.to(DEVICE) - MU3) if centered else X9t.clone().to(DEVICE)
    Xb0 = X9BIN.clone().to(DEVICE); P = EYE.clone(); rank = 0; curve = []
    erase = (lambda X, P_: MU3 + (X - MU3) @ P_.T) if centered else (lambda X, P_: X @ P_.T)
    for target in ranks:
        while rank < target:
            lin = fit_digit_probe(Xtr, ydig, seed=rank) if digits else fit_probe(Xtr, ytr, seed=rank)
            Q, _ = torch.linalg.qr(lin.weight.detach().T)
            Pk = EYE - Q @ Q.T
            P = Pk @ P; Xtr = Xtr @ Pk.T; rank += Q.shape[1]
        if digits:
            z = fit_digit_probe(Xtr, ydig, seed=1000 + rank)(Xtr)
            acc = [round(float((z[:, 10*k:10*k+10].argmax(1) == ydig[:, k]).float().mean()), 3) for k in range(3)]
        else:
            acc = round(float((fit_probe(Xtr, ytr, seed=1000 + rank)(Xtr).argmax(1) == ytr).float().mean()), 4)
        g = torch.linalg.qr(torch.randn(D_D, rank, device=DEVICE,
                            generator=torch.Generator(device=DEVICE).manual_seed(rank)))[0]
        Pr = EYE - g @ g.T
        row = {"rank": int(rank), "probe_train_acc_after (EXP28's number)": acc,
               "probe_heldout_acc_after": heldout_probe_acc(erase(X9t.to(DEVICE), P), erase(Xb0, P), "digits" if digits else "lead"),
               "mean_shift_norm": {"answer": round(float((MU3 - MU3 @ P.T).norm()), 3) if not centered else 0.0,
                                   "random": round(float((MU3 - MU3 @ Pr.T).norm()), 3) if not centered else 0.0}}
        for name in ERASE_MAPS:
            ea, da = _erased(name, erase(Xb0, P)); er, dr_ = _erased(name, erase(Xb0, Pr))
            row[name] = {"answer_erased": fmt(wilson_bools(ea)), "random_erased": fmt(wilson_bools(er)),
                         "answer_erased_per_digit": da, "random_erased_per_digit": dr_}
            PER_ITEM[f"{variant}/{name}/rank{rank}"] = [int(x) for x in ea]
        curve.append(row)
        print(f"  {variant} rank {rank}: probe train {acc} / held-out {row['probe_heldout_acc_after']} | mean shift {row['mean_shift_norm']}")
        for name in ERASE_MAPS:
            print(f"      {name:14s} answer-erased {row[name]['answer_erased']} {row[name]['answer_erased_per_digit']} | "
                  f"random {row[name]['random_erased']} {row[name]['random_erased_per_digit']}")
    D[variant] = {"curve": curve,
                  "mlp_heldout_after_deepest_rank": mlp_suite(erase(X9t.to(DEVICE), P), erase(Xb0, P))}
    print(f"  {variant} MLP probe after erasure (held-out on bin): {D[variant]['mlp_heldout_after_deepest_rank']}")
    del Xtr, Xb0, P; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
del EYE
RESULTS["D_answer_erasure"] = D
print("SANITY (uncentered leading-digit erasure of exp28_3k seed 0 vs GitHub EXP28 EXP-C):")
for row in D["D1_leading_digit_uncentered_EXP28"]["curve"]:
    r = row["rank"]
    if r in REF["exp28_3k_erasure_seed0"] and "exp28_3k_s0" in row:
        check(f"erasure rank {r} answer", float(row["exp28_3k_s0"]["answer_erased"].split()[0]), REF["exp28_3k_erasure_seed0"][r], 0.015, warn=0.03)
        check(f"erasure rank {r} random", float(row["exp28_3k_s0"]["random_erased"].split()[0]), REF["exp28_3k_erasure_random_seed0"][r], 0.015, warn=0.03)
print(f"  [{time.time()-t0:.0f}s]"); save("D")

  unerased: {'exp28_3k_s0': {'whole': 0.217, 'per_digit': {'hundreds': 0.919, 'tens': 0.313, 'ones': 0.588}}, 'paper_6ep': {'whole': 0.278, 'per_digit': {'hundreds': 0.887, 'tens': 0.375, 'ones': 0.636}}, 'exp28_18k_s0': {'whole': 0.586, 'per_digit': {'hundreds': 0.954, 'tens': 0.647, 'ones': 0.856}}}
  probes before erasure (held-out on bin): {'donor_linear_lead': 0.89, 'donor_linear_digits': [0.887, 0.165, 0.649], 'donor_mlp': {'leading_digit': 0.939, 'hundreds': 0.936, 'tens': 0.316, 'ones': 0.778}, 'recipient_own_L11_mlp (operands baseline)': {'leading_digit': 0.923, 'hundreds': 0.922, 'tens': 0.204, 'ones': 0.685}}
  D1_leading_digit_uncentered_EXP28 rank 9: probe train 0.825 / held-out 0.738 | mean shift {'answer': 1.468, 'random': 0.462}
      exp28_3k_s0    answer-erased 0.091 [0.071, 0.116] {'hundreds': 0.518, 'tens': 0.168, 'ones': 0.57} | random 0.217 [0.187, 0.251] {'hundreds': 0.919, 'tens': 0.312, 'ones': 0.585}
      paper_6ep      answer-erased 0.115 [0.092, 0.142] {'hu

In [19]:
# === CELL 14: PART E — prompt swap, per digit (port of Part D in Stitch_Llama_AnswerBottleneck) ===
# Recipient reads P' = (a, b, c + delta); the graft is built from the donor on P. A digit that matches P
# came through the graft; a digit that matches P' was computed by the recipient from its own prompt.
# NULLS for "the recipient computes a digit": PROXIMITY (reads P, graft from P, scored against y': how
# often the graft alone lands on y''s digit) and CHANCE (MISMATCH outputs vs a third problem's answer).
# A digit is recipient-computed if MISMATCH-vs-y' is well above BOTH.
# REFERENCES (not nulls; they measure the recipient's own computing on P'): NATIVE (no graft) and
# SHUFFLED (graft from an unrelated problem). 'vs_native' = MISMATCH-vs-y' / NATIVE-vs-y': ~0 means the
# graft suppresses the recipient's own digit, ~1 it computes as much as alone, >1 the graft helps it.
# const_* (one vector, no donor information) is the cleanest "graft slot filled, nothing sent" reference.
# MATCHED (donor on P') is also reported on P' outside the map's training set (P' or its twin can be a
# training problem, which inflates MATCHED).
t0 = time.time()
_EXPR_RE = re.compile(r"^(\d+) \* (\d+) \+ (\d+)$")
DELTAS = [d for k in range(1, ARITH["c_hi"]) for d in (k, -k)]
_NT = {}
def _ntok(x):
    if x not in _NT: _NT[x] = len(tokenizer(str(x), add_special_tokens=False).input_ids)
    return _NT[x]
def build_swap_items(rule, seed=P_SHUFFLE_SEED):
    rng = random.Random(seed); items, drop = [], 0
    for i in unsolv:
        mm = _EXPR_RE.match(evalp[i]["expr"])
        if mm is None: drop += 1; continue
        a, b, c = map(int, mm.groups()); y = a * b + c; dy = _digits(y); cands = []
        for dlt in DELTAS:
            c2 = c + dlt; y2 = a * b + c2
            if not (1 <= c2 <= ARITH["c_hi"] and ARITH["ans_min"] <= y2 <= ARITH["ans_max"]): continue
            if _ntok(c2) != _ntok(c): continue
            d2 = _digits(y2)
            if rule == "leading":
                if d2[0] == dy[0]: continue
            elif d2[2] == dy[2] or abs(d2[1] - dy[1]) < 2 or y2 % 10 == 0: continue
            cands.append((dlt, c2, y2))
        if rule == "far": rng.shuffle(cands)
        pick = None
        for dlt, c2, y2 in cands:
            ids2, tok2 = _aencode(tokenizer, f"{a} * {b} + {c2}", y2)
            if ids2 is None or tok2 == evalp[i]["tok"]: continue
            if ids2.numel() != evalp[i]["ids"].numel(): continue
            pick = dict(idx=i, y=y, y2=y2, ans=y2, expr=f"{a} * {b} + {c2}", ids=ids2, tok=tok2,
                        stale_tok=evalp[i]["tok"]); break
        if pick is None: drop += 1
        else: items.append(pick)
    return items, drop
def _score(tops, items, ks, ans_of):
    out = {"whole": [tokenizer.decode([tops[k]]).strip() == str(ans_of(k)) for k in ks]}
    for n, nm in enumerate(("hundreds", "tens", "ones")):
        kn = [k for k in ks if _digits(items[k]["y"])[n] != _digits(items[k]["y2"])[n]]
        out[nm] = [_dig_of(tops[k], n) == _digits(ans_of(k))[n] for k in kn]
    return out
def _r(d): return {k: rate(v) for k, v in d.items()}

SWAP_MAPS = [n for n in ("exp28_3k_s0", "exp28_3k_s1", "paper_6ep", "const_3k", "exp28_18k_s0", "exp28_18k_s1", "const_18k") if n in MAPS]
E = {}
for rule in ("far", "leading"):
    items, drop = build_swap_items(rule)
    n = len(items); src = [p["idx"] for p in items]
    if n == 0: E[rule] = {"n": 0}; continue
    X9P, donor_top_P = states_last_and_top(model_d, L_D, items)              # donor on P'
    rng = np.random.default_rng(P_SHUFFLE_SEED); shuf_src = []
    for k in range(n):
        while True:
            s_ = int(rng.integers(0, len(evalp)))
            if s_ != src[k] and evalp[s_]["ans"] not in (items[k]["y"], items[k]["y2"]): break
        shuf_src.append(s_)
    third = [[None] * n for _ in range(CHANCE_DRAWS)]
    for d in range(CHANCE_DRAWS):
        for k in range(n):
            while True:
                u = int(rng.integers(0, len(evalp)))
                if evalp[u]["ans"] not in (items[k]["y"], items[k]["y2"]): break
            third[d][k] = evalp[u]["ans"]
    orig_items = [evalp[i] for i in src]
    nat = run_tops(None, items)
    KS = [k for k in range(n) if nat[k] != items[k]["tok"]]
    y_of, y2_of, u_of = (lambda k: items[k]["y"]), (lambda k: items[k]["y2"]), (lambda k: evalp[shuf_src[k]]["ans"])
    res = {"n_kept": n, "n_scored": len(KS),
           "NATIVE_vs_y2 (whole is 0 by construction: P' is natively unsolvable)": _r(_score(nat, items, KS, y2_of)), "maps": {}}
    NAT = _r(_score(nat, items, KS, y2_of))
    for name in SWAP_MAPS:
        f = fn_of(MAPS[name])
        t_mis, t_mat = run_tops(f, items, X9e[src]), run_tops(f, items, X9P)
        t_sh, t_o = run_tops(f, items, X9e[shuf_src]), run_tops(f, orig_items, X9e[src])
        mis_y, mis_y2 = _score(t_mis, items, KS, y_of), _score(t_mis, items, KS, y2_of)
        o_y = [t_o[k] == items[k]["stale_tok"] for k in KS]
        ch = collections.defaultdict(list)
        for d in range(CHANCE_DRAWS):
            for key, v in _r(_score(t_mis, items, KS, lambda k, d=d: third[d][k])).items(): ch[key].append(v)
        pm = mcnemar_exact(o_y, mis_y["whole"])
        _ex = {p["expr"] for p in TRAIN_OF[name][0]}
        KS_OOT = [k for k in KS if items[k]["expr"] not in _ex and _twin(items[k]["expr"]) not in _ex]
        _m2 = _r(mis_y2)
        res["maps"][name] = {
            "ORIGINAL_vs_y": _r(_score(t_o, items, KS, y_of)), "PROXIMITY_null_vs_y2": _r(_score(t_o, items, KS, y2_of)),
            "MISMATCH_vs_y": _r(mis_y), "MISMATCH_vs_y2": _r(mis_y2),
            "MATCHED_vs_y2": _r(_score(t_mat, items, KS, y2_of)),
            "MATCHED_vs_y2_Pprime_not_in_training": {"n": len(KS_OOT), **_r(_score(t_mat, items, KS_OOT, y2_of))},
            "MISMATCH_vs_y2_over_NATIVE": {k: (round(_m2[k] / NAT[k], 2) if NAT.get(k) and _m2.get(k) is not None else None)
                                           for k in ("hundreds", "tens", "ones")},
            "SHUFFLED_vs_its_donor": _r(_score(t_sh, items, KS, u_of)), "SHUFFLED_vs_y2": _r(_score(t_sh, items, KS, y2_of)),
            "CHANCE": {k: round(float(np.nanmean([x for x in v if x is not None])), 3) for k, v in ch.items()},
            "paired_original_vs_mismatch_whole": {"kept_share": (round(rate(mis_y["whole"]) / rate(o_y), 3) if rate(o_y) else None),
                                                  "lost": pm["n10"], "gained": pm["n01"], "p": pm["p"]}}
    E[rule] = res
    print(f"\n  PROMPT SWAP '{rule}': kept {n}, scored {len(KS)} | NATIVE vs y' (reference): {NAT}")
    print(f"  each cell: MISMATCH output's digit = y (donor's problem) / = y' (recipient's prompt);  nulls for y': PROXIMITY, CHANCE")
    print(f"  {'map':14s} {'ORIG':>6s} | {'whole y/yp':>11s} | {'tens y/yp':>11s} | {'ones y/yp':>11s} || ones yp nulls: prox / chance"
          f" | ones yp / NATIVE | kept")
    for name, r in res["maps"].items():
        my, my2 = r["MISMATCH_vs_y"], r["MISMATCH_vs_y2"]
        c = lambda k: f"{my[k]}/{my2[k]}"
        print(f"  {name:14s} {r['ORIGINAL_vs_y']['whole']!s:>6s} | {c('whole'):>11s} | {c('tens'):>11s} | {c('ones'):>11s} || "
              f"{r['PROXIMITY_null_vs_y2']['ones']} / {r['CHANCE']['ones']} | {r['MISMATCH_vs_y2_over_NATIVE']['ones']} | "
              f"{r['paired_original_vs_mismatch_whole']['kept_share']}")
RESULTS["E_prompt_swap"] = E
print("SANITY (prompt-swap item counts vs the earlier Llama run):")
for rule, (kept, scored) in REF["swap_counts"].items():
    if rule in E and "n_kept" in E[rule]:
        check(f"swap {rule} kept", E[rule]["n_kept"], kept, 0 if not SMOKE_TEST else 10**9, warn=2)
        check(f"swap {rule} scored", E[rule]["n_scored"], scored, 0 if not SMOKE_TEST else 10**9, warn=2)
print("SANITY (the paper map under the prompt swap vs the earlier Llama run, task_3k_seed0):")
for rule, tg in REF["paper_6ep_swap"].items():
    r = E.get(rule, {}).get("maps", {}).get("paper_6ep")
    if r is None: continue
    check(f"swap {rule} ORIGINAL whole", r["ORIGINAL_vs_y"]["whole"], tg["ORIGINAL_y_whole"], 0.012, warn=0.025)
    check(f"swap {rule} MISMATCH whole vs y", r["MISMATCH_vs_y"]["whole"], tg["MISMATCH_y_whole"], 0.012, warn=0.025)
    if "MISMATCH_y2_ones" in tg:
        check(f"swap {rule} MISMATCH ones vs y'", r["MISMATCH_vs_y2"]["ones"], tg["MISMATCH_y2_ones"], 0.012, warn=0.025)
print(f"  [{time.time()-t0:.0f}s]"); save("E")


  PROMPT SWAP 'far': kept 609, scored 598 | NATIVE vs y' (reference): {'whole': 0.0, 'hundreds': 0.279, 'tens': 0.057, 'ones': 0.206}
  each cell: MISMATCH output's digit = y (donor's problem) / = y' (recipient's prompt);  nulls for y': PROXIMITY, CHANCE
  map              ORIG |  whole y/yp |   tens y/yp |   ones y/yp || ones yp nulls: prox / chance | ones yp / NATIVE | kept
  exp28_3k_s0     0.207 | 0.132/0.033 |  0.273/0.14 |  0.37/0.176 || 0.038 / 0.097 | 0.85 | 0.638
  exp28_3k_s1     0.244 | 0.166/0.025 | 0.291/0.132 | 0.441/0.134 || 0.03 / 0.1 | 0.65 | 0.68
  paper_6ep       0.266 | 0.201/0.017 | 0.328/0.115 | 0.512/0.092 || 0.035 / 0.101 | 0.45 | 0.756
  const_3k        0.194 | 0.003/0.144 | 0.095/0.217 |  0.033/0.61 || 0.035 / 0.096 | 2.96 | 0.015
  exp28_18k_s0    0.587 |   0.537/0.0 | 0.615/0.062 | 0.816/0.028 || 0.018 / 0.102 | 0.14 | 0.915
  exp28_18k_s1    0.567 | 0.507/0.003 |  0.562/0.06 | 0.836/0.018 || 0.008 / 0.101 | 0.09 | 0.894
  const_18k       0.201 | 0.003/0.14

In [20]:
# === CELL 15: summary ===
RESULTS["SANITY"] = SANITY
_st = collections.Counter(v["status"] for v in SANITY.values())
print(f"SANITY: {_st['PASS']} PASS, {_st['WARN']} WARN, {_st['CHECK']} CHECK of {len(SANITY)}")
for k, v in SANITY.items():
    if v["status"] != "PASS": print(f"   {v['status']}:", k, v)
print("\nA. ROOM TO SPARE (frozen map; conferral / mean CE)")
for name, row in RESULTS["A_room_to_spare"].items():
    tr = row.get("train/donor_right_recipient_wrong (bin-like)", {}); dw = row.get("train/donor_wrong_recipient_wrong", {})
    print(f"  {name:14s} train bin-like {tr.get('conferral')} / CE {tr.get('mean_CE')} (donor CE {tr.get('donor_own_CE_on_gold')}) | "
          f"train donor-wrong {dw.get('conferral')} / CE {dw.get('mean_CE')} (donor CE {dw.get('donor_own_CE_on_gold')}; emits donor's "
          f"wrong answer {dw.get('emits_donors_own_wrong_answer')}) | EVAL BIN {row['eval_bin']['conferral']} / CE {row['eval_bin']['mean_CE']}"
          f" | eval twin-in-train / not: {row['eval_bin/twin_in_training']['conferral']} / {row['eval_bin/no_twin_in_training']['conferral']}")
print("\nB. SHUFFLE (own answer / the other problem's answer)")
for name, v in RESULTS["B_shuffle"].items():
    if isinstance(v, dict): print(f"  {name:14s} {v['own_answer']}  /  {v['partner_answer']}")
print("\nC. SVD TRUNCATION, share of full-rank conferral kept (eval bin | train bin-like) at ranks 16 / 32 / 64")
for name, v in RESULTS["C_svd_truncation"].items():
    for mode in ("W", "dW"):
        br = v[f"truncate_{mode}"]
        print(f"  {name:14s} {mode:2s} " + "  ".join(
            f"r{r}: {br[r]['share_of_full_kept'].get('eval_bin')} | {br[r]['share_of_full_kept'].get('train/donor_right_recipient_wrong (bin-like)')}"
            for r in (16, 32, 64) if r in br) + f"   (full: eval {br['full']['eval_bin']})")
print("\nD. ERASURE at the deepest rank (answer-erased vs random; per-digit after erasure)")
DD = RESULTS["D_answer_erasure"]
print(f"  unerased: {DD['unerased']}")
print(f"  probes BEFORE (held-out on bin): {DD['probes_before']}")
for variant, v in DD.items():
    if not isinstance(v, dict) or "curve" not in v: continue
    last = v["curve"][-1]
    print(f"  {variant} rank {last['rank']}: held-out linear probe after {last['probe_heldout_acc_after']}")
    for n in ERASE_MAPS:
        print(f"      {n:14s} {last[n]['answer_erased']} {last[n]['answer_erased_per_digit']}  vs random {last[n]['random_erased']}")
    print(f"      MLP probe AFTER (held-out on bin): {v['mlp_heldout_after_deepest_rank']}")
print("\nE. PROMPT SWAP ('far'): MISMATCH digit = y / = y' ; nulls for y' (proximity, chance); y'/NATIVE")
for name, r in RESULTS["E_prompt_swap"].get("far", {}).get("maps", {}).items():
    my, my2 = r["MISMATCH_vs_y"], r["MISMATCH_vs_y2"]
    print(f"  {name:14s} whole {my['whole']}/{my2['whole']}  tens {my['tens']}/{my2['tens']}  ones {my['ones']}/{my2['ones']} | "
          f"nulls ones {r['PROXIMITY_null_vs_y2']['ones']}/{r['CHANCE']['ones']} | ones/NATIVE {r['MISMATCH_vs_y2_over_NATIVE']['ones']} | "
          f"kept {r['paired_original_vs_mismatch_whole']['kept_share']}")
save("summary")

SANITY: 46 PASS, 0 WARN, 0 CHECK of 46

A. ROOM TO SPARE (frozen map; conferral / mean CE)
  exp28_3k_s0    train bin-like 0.748 [0.719, 0.775] / CE 0.9756 (donor CE 0.879) | train donor-wrong 0.606 [0.584, 0.627] / CE 1.5662 (donor CE 3.0755; emits donor's wrong answer 0.027 [0.021, 0.035]) | EVAL BIN 0.217 [0.187, 0.251] / CE 2.8573 | eval twin-in-train / not: 0.432 [0.330, 0.541] / 0.185 [0.155, 0.220]
  exp28_3k_s1    train bin-like 0.747 [0.718, 0.774] / CE 0.9738 (donor CE 0.879) | train donor-wrong 0.613 [0.591, 0.634] / CE 1.4206 (donor CE 3.0755; emits donor's wrong answer 0.034 [0.027, 0.043]) | EVAL BIN 0.246 [0.214, 0.281] / CE 2.6881 | eval twin-in-train / not: 0.444 [0.341, 0.553] / 0.217 [0.184, 0.253]
  paper_6ep      train bin-like 0.943 [0.926, 0.956] / CE 0.2453 (donor CE 0.879) | train donor-wrong 0.893 [0.879, 0.906] / CE 0.4038 (donor CE 3.0755; emits donor's wrong answer 0.007 [0.004, 0.012]) | EVAL BIN 0.278 [0.244, 0.314] / CE 2.8207 | eval twin-in-train / not:

## Reading the results

**S. Sanity.** Lines say PASS, WARN or CHECK. WARN = a couple of items off, which a bf16 argmax tie or
a different GPU can cause; fine. A CHECK on the bin or swap counts means the problems differ from
the paper's: stop there. A CHECK on a loss or conferral value means the run isn't reproducing; look
at it before reading anything else.

**A. Room to spare (the precondition).** Read the frozen map's training rows next to its eval bin.
- *Bin-like training problems* (donor right, recipient wrong: the eval bin's filter). Conferral well
  below 1 and CE well above 0 = the map didn't fit even training problems whose answer the donor
  gets right at its output. The donor finishes its computation after layer 17, so better transfer,
  including computation, could still have lowered that loss.
- *Donor-wrong training problems* are the decisive subset: fitting them needs something beyond the
  donor's own output. Three numbers:
  - conferral (how often it fit the gold label anyway);
  - "emits donor's own wrong answer" (how often it passed on the donor's mistake);
  - the map's CE against the donor's own CE on the gold token. A map below the donor's CE is fitting
    labels the donor's output doesn't have: memorisation or computation. Low conferral there means it
    was trained on those labels and still mostly couldn't produce them.
- The 2-epoch maps' loss is still falling fast, so their "room" is partly undertraining. The cleanest
  precondition is `paper_6ep`, especially if its per-epoch loss has flattened.
- *Twin split.* Eval problems whose operand-swap twin is a training problem can be near-lookup. If
  the no-twin column is well below the twin column, the train-vs-eval gap understates memorisation.
  This matters most at 18k, where twins are common.
- `const_*` (one vector for every problem, same number of Adam steps at 3k and 18k) is what the
  recipient gets with no donor information: read conferral against it, not against 0.

**Then, despite the room, did anything beyond the answer transfer?**
- **B. Shuffle.** Own answer near 0 AND the other problem's answer near the map's conferral → the
  output is whatever answer the donor state carries.
- **C. SVD.** Read the *share of full-rank conferral kept*, training vs eval.
  - Few directions keeping most of the *eval* share → the map had about 2,000 spare directions it
    didn't use for anything that generalises.
  - Cutting the rest costing far more of the *training* share → those directions were memorising
    training problems.
  - `truncate_dW` is the cleaner version (what training added, with the ridge start kept).
- **D. Erasure.** Answer-erased whole-answer conferral near 0 while random-erased stays at baseline →
  conferral depends on the answer directions in the donor state.
  - *D1* reproduces EXP28, but on uncentered states it also shifts the mean; compare `mean_shift_norm`
    for answer vs random.
  - *D1c / D2c* are centered, so no mean shift. D2c removes the tens and ones directions too.
  - Check the per-digit columns: if tens or ones survive erasure while the hundreds collapses, those
    digits are arriving some other way, or being supplied by the recipient.
  - MLP check: leftover nonlinear answer information is indicated only if the MLP after erasure is well
    above BOTH the held-out linear probe after erasure and the recipient's-own-state MLP baseline
    (which measures an MLP computing digits from operand information).
- **E. Prompt swap.** For each digit, compare MISMATCH-vs-y′ with its nulls (PROXIMITY, CHANCE).
  - At the nulls, with the digit following y → that digit came through the graft.
  - Clearly above both → the recipient computes that digit from its own prompt. Report it per digit,
    as for the rank-27 map.
  - `ones yp / NATIVE`: ~0 means the graft suppresses the recipient's own digit; ~1 it computes as much
    as alone; above 1 the graft helps it.
  - `const_*` shows the recipient's own computing with the graft slot filled and nothing sent.

**Limits.**
- No single control proves the absence of computation. Together they show the conferral is carried
  by the donor's answer content and doesn't use the recipient's own arithmetic on its prompt.
- E can't see computation that depends only on a and b, which don't change in P′.
- D's erasure probes are fit on 3k donor states and also applied to the 18k map.
- Single seed except where two seeds are listed.